In [ ]:
!apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import threading
import subprocess

# Start Ollama service in background

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

In [ ]:
!ollama pull gemma3:1b

In [ ]:
!pip install dspy-ai

In [ ]:
import dspy
import requests

In [ ]:
def setup_ollama():
  try:
    requests.get("http://localhost:11434/api/tags")
    print("Ollama está rodando")
  except:
    print("Ollama não está rodando. Tente dar start no ollama serve")
    return False

setup_ollama()

Ollama está rodando


In [ ]:
dspy.settings.configure(
    lm=dspy.LM(
        model="ollama/gemma3:1b",
        api_base="http://localhost:11434",
        max_tokens=500,
        temperature=0.7,
    )
)
print("Connected to Ollama (Gemma3 1b)")

Connected to Ollama (Gemma3 1b)


In [ ]:
import sqlite3

def create_supermarket_database():
    conn = sqlite3.connect("supermarket.db")
    c = conn.cursor()

    # Tabelas
    c.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id INTEGER PRIMARY KEY, name TEXT, category TEXT,
            price REAL, stock INTEGER
        )
    """)
    c.execute("""
        CREATE TABLE IF NOT EXISTS suppliers (
            id INTEGER PRIMARY KEY, name TEXT,
            contact TEXT, city TEXT
        )
    """)
    c.execute("""
        CREATE TABLE IF NOT EXISTS sales (
            id INTEGER PRIMARY KEY, product_id INTEGER,
            quantity INTEGER, sale_date TEXT, total REAL
        )
    """)

    # Produtos
    c.executemany("INSERT OR IGNORE INTO products VALUES (?, ?, ?, ?, ?)", [
        (1, "Arroz 5kg",       "Grãos",      25.90, 200),
        (2, "Feijão 1kg",      "Grãos",       8.50, 150),
        (3, "Leite Integral",  "Laticínios",  5.99, 300),
        (4, "Queijo Mussarela","Laticínios", 35.00,  80),
        (5, "Frango 1kg",      "Carnes",     18.90, 120),
        (6, "Detergente",      "Limpeza",     2.99, 400),
    ])

    # Fornecedores
    c.executemany("INSERT OR IGNORE INTO suppliers VALUES (?, ?, ?, ?)", [
        (1, "Distribuidora Grãos Sul", "grãos@sul.com",      "São Paulo"),
        (2, "Laticínios Bela Vista",  "contato@belavista.com","Campinas"),
        (3, "Frigorífico Central",    "vendas@frigoc.com",   "Taubaté"),
    ])

    # Vendas
    c.executemany("INSERT OR IGNORE INTO sales VALUES (?, ?, ?, ?, ?)", [
        (1, 1, 10, "2025-02-01", 259.00),
        (2, 3, 25, "2025-02-01", 149.75),
        (3, 5,  8, "2025-02-02", 151.20),
        (4, 2, 15, "2025-02-03", 127.50),
        (5, 4,  5, "2025-02-03", 175.00),
    ])

    conn.commit()
    conn.close()
    print("Banco do supermercado criado!")

create_supermarket_database()

Banco do supermercado criado!


In [ ]:
class GenerateSQL(dspy.Signature):
    """
    Generate a valid SQLite SELECT query for a supermarket database.

    Rules:
    - Return only SQL.
    - Use only SELECT queries.
    - Do not use INSERT, UPDATE, DELETE, DROP, ALTER or CREATE.
    - Use SQLite syntax.
    - Use JOIN when the question needs data from more than one table.

    Schema:

    products(id, name, category, price, stock)
    suppliers(id, name, contact, city)
    sales(id, product_id, quantity, sale_date, total)

    Relationship:
    sales.product_id references products.id.
    """

    question = dspy.InputField(desc="Natural language question in Portuguese")
    sql_query = dspy.OutputField(desc="A valid SQLite SELECT query")

In [ ]:
def is_safe_sql(sql):
    sql_clean = sql.strip().lower()

    forbidden_words = [
        "insert", "update", "delete", "drop", "alter",
        "create", "replace", "truncate", "pragma"
    ]

    if not sql_clean.startswith("select"):
        return False

    for word in forbidden_words:
        if word in sql_clean:
            return False

    return True

In [ ]:
class SQLGenerator(dspy.Module):
    """Generate and execute SQL queries with error refinement."""

    def __init__(self):
        super().__init__()
        self.generator = dspy.ChainOfThought(GenerateSQL)
        self.refiner = dspy.ChainOfThought(
            "question, sql_query, error -> refined_sql"
        )

    def forward(self, question, conn):
        output = self.generator(question=question)
        sql = output.sql_query.strip()

        if not is_safe_sql(sql):
            return dspy.Prediction(
                sql_query=sql,
                results=None,
                error="Consulta bloqueada por segurança. Apenas SELECT é permitido."
            )

        try:
            results = conn.execute(sql).fetchall()
            return dspy.Prediction(
                sql_query=sql,
                results=results,
                error=None
            )

        except Exception as e:
            refined = self.refiner(
                question=question,
                sql_query=sql,
                error=str(e)
            )

            refined_sql = refined.refined_sql.strip()

            if not is_safe_sql(refined_sql):
                return dspy.Prediction(
                    sql_query=refined_sql,
                    results=None,
                    error="Consulta refinada bloqueada por segurança. Apenas SELECT é permitido."
                )

            try:
                results = conn.execute(refined_sql).fetchall()
                return dspy.Prediction(
                    sql_query=refined_sql,
                    results=results,
                    error=None
                )

            except Exception as e2:
                return dspy.Prediction(
                    sql_query=refined_sql,
                    results=None,
                    error=str(e2)
                )

In [ ]:

    def create_train_examples():
      """exemplos de treinamento para few-shot learning."""
      return [
        dspy.Example(
            question="Quais produtos têm estoque abaixo de 100?",
            sql_query="SELECT name, stock FROM products WHERE stock < 100"
        ).with_inputs("question"),

        dspy.Example(
            question="Qual o produto mais caro?",
            sql_query="SELECT name, price FROM products ORDER BY price DESC LIMIT 1"
        ).with_inputs("question"),

        dspy.Example(
            question="Quais produtos são da categoria Laticínios?",
            sql_query="SELECT name, price FROM products WHERE category = 'Laticínios'"
        ).with_inputs("question"),

        dspy.Example(
            question="Qual foi o total de vendas em fevereiro de 2025?",
            sql_query="SELECT SUM(total) FROM sales WHERE sale_date LIKE '2025-02%'"
        ).with_inputs("question"),

        dspy.Example(
            question="Quais fornecedores são de São Paulo?",
            sql_query="SELECT name, contact FROM suppliers WHERE city = 'São Paulo'"
        ).with_inputs("question"),

        dspy.Example(
            question="Quais produtos têm preço maior que 10 reais?",
            sql_query="SELECT name, price FROM products WHERE price > 10"
        ).with_inputs("question"),

        dspy.Example(
            question="Quais produtos estão na categoria Grãos?",
            sql_query="SELECT name, price FROM products WHERE category = 'Grãos'"
        ).with_inputs("question"),

        dspy.Example(
            question="Qual é o estoque total de produtos?",
            sql_query="SELECT SUM(stock) FROM products"
        ).with_inputs("question"),

        dspy.Example(
            question="Quantas vendas foram registradas?",
            sql_query="SELECT COUNT(*) FROM sales"
        ).with_inputs("question"),

        dspy.Example(
            question="Quais produtos foram vendidos no dia 2025-02-03?",
            sql_query="""
SELECT products.name, sales.quantity, sales.total
FROM sales
JOIN products ON sales.product_id = products.id
WHERE sales.sale_date = '2025-02-03'
"""
        ).with_inputs("question"),
    ]

In [ ]:
def create_test_examples():
    return [
        dspy.Example(
            question="Liste os produtos com estoque maior que 100.",
            sql_query="SELECT name, stock FROM products WHERE stock > 100"
        ).with_inputs("question"),

        dspy.Example(
            question="Qual fornecedor é de Taubaté?",
            sql_query="SELECT name, contact FROM suppliers WHERE city = 'Taubaté'"
        ).with_inputs("question"),

        dspy.Example(
            question="Qual produto tem o menor preço?",
            sql_query="SELECT name, price FROM products ORDER BY price ASC LIMIT 1"
        ).with_inputs("question"),

        dspy.Example(
            question="Qual foi o valor médio das vendas?",
            sql_query="SELECT AVG(total) FROM sales"
        ).with_inputs("question"),

        dspy.Example(
            question="Mostre o nome dos produtos vendidos com suas quantidades.",
            sql_query="""
SELECT products.name, sales.quantity
FROM sales
JOIN products ON sales.product_id = products.id
"""
        ).with_inputs("question"),
    ]

In [ ]:
conn = sqlite3.connect("supermarket.db")

sql_gen = SQLGenerator()

perguntas = [
    "Quais produtos têm estoque abaixo de 100?",
    "Qual foi o total de vendas em fevereiro de 2025?",
    "Quais produtos são da categoria Laticínios?",
]

for pergunta in perguntas:
    result = sql_gen(question=pergunta, conn=conn)
    print(f"Pergunta: {pergunta}")
    print(f"SQL: {result.sql_query}")
    print(f"Resultado: {result.results}")
    print(f"Erro: {result.error}\n")

conn.close()

Pergunta: Quais produtos têm estoque abaixo de 100?
SQL: SELECT id, name FROM products WHERE stock < 100;
Resultado: [(4, 'Queijo Mussarela')]
Erro: None

Pergunta: Qual foi o total de vendas em fevereiro de 2025?
SQL: SELECT SUM(total) FROM sales WHERE sale_date BETWEEN '2025-02-01' AND '2025-02-28'
Resultado: [(862.45,)]
Erro: None

Pergunta: Quais produtos são da categoria Laticínios?
SQL: SELECT name FROM products WHERE category = 'Laticínios';
Resultado: [('Leite Integral',), ('Queijo Mussarela',)]
Erro: None



In [ ]:
# Setup generator with few-shot examples
generator = SQLGenerator()
generator.generator.demos = create_train_examples()
generator.generator.demos = create_test_examples()
print("✓ Gerador configura com alguns exemplos")


✓ Gerador configura com alguns exemplos


In [ ]:
def main():
    print("\n" + "="*60)
    print("Text-to-SQL com DSPy & Ollama")
    print("="*60 + "\n")

    # Inicializa conexão com Ollama
    setup_ollama()

    # Cria banco do supermercado
    create_supermarket_database()

    # Configura o gerador com exemplos few-shot
    generator = SQLGenerator()
    generator.generator.demos = create_train_examples()
    generator.generator.demos = create_test_examples()
    print("✓ Gerador configurado com exemplos few-shot\n")

    # Conecta ao banco
    conn = sqlite3.connect("supermarket.db")

    # Perguntas de teste
    test_questions = [
        "Quais produtos têm estoque abaixo de 100?",
        "Qual foi o total de vendas em fevereiro de 2025?",
        "Quais produtos são da categoria Laticínios?",
        "Qual o produto mais caro?",
        "Quais fornecedores são de São Paulo?",
    ]

    print("="*60)
    print("TESTANDO TEXT-TO-SQL")
    print("="*60 + "\n")

    for i, question in enumerate(test_questions, 1):
        print(f"Query {i}: {question}")
        result = generator.forward(question, conn)
        print(f"SQL: {result.sql_query}")
        print(f"Resultado: {result.results if not result.error else f'Erro: {result.error}'}\n")

    conn.close()
    print("✓ Concluído!")

if __name__ == "__main__":
    main()


Text-to-SQL com DSPy & Ollama

Ollama está rodando
Banco do supermercado criado!
✓ Gerador configurado com exemplos few-shot

TESTANDO TEXT-TO-SQL

Query 1: Quais produtos têm estoque abaixo de 100?


2026/05/07 13:24:27 WARNING dspy.primitives.module: Calling module.forward(...) on SQLGenerator directly is discouraged. Please use module(...) instead.
2026/05/07 13:24:28 WARNING dspy.primitives.module: Calling module.forward(...) on SQLGenerator directly is discouraged. Please use module(...) instead.


SQL: SELECT id, name FROM products WHERE stock < 100;
Resultado: [(4, 'Queijo Mussarela')]

Query 2: Qual foi o total de vendas em fevereiro de 2025?


2026/05/07 13:24:28 WARNING dspy.primitives.module: Calling module.forward(...) on SQLGenerator directly is discouraged. Please use module(...) instead.


SQL: SELECT SUM(total) FROM sales WHERE sale_date BETWEEN '2025-02-01' AND '2025-02-28'
Resultado: [(862.45,)]

Query 3: Quais produtos são da categoria Laticínios?


2026/05/07 13:24:29 WARNING dspy.primitives.module: Calling module.forward(...) on SQLGenerator directly is discouraged. Please use module(...) instead.


SQL: SELECT name FROM products WHERE category = 'Laticínios';
Resultado: [('Leite Integral',), ('Queijo Mussarela',)]

Query 4: Qual o produto mais caro?


2026/05/07 13:24:30 WARNING dspy.primitives.module: Calling module.forward(...) on SQLGenerator directly is discouraged. Please use module(...) instead.


SQL: SELECT MAX(price) FROM products;
Resultado: [(35.0,)]

Query 5: Quais fornecedores são de São Paulo?
SQL: SELECT name, city FROM suppliers WHERE city = 'São Paulo';
Resultado: [('Distribuidora Grãos Sul', 'São Paulo')]

✓ Concluído!


In [ ]:
print("="*60)
print("MODO INTERATIVO - digite 'sair' para encerrar")
print("="*60 + "\n")

conn = sqlite3.connect("supermarket.db")
generator = SQLGenerator()
generator.generator.demos = create_train_examples()
generator.generator.demos = create_test_examples()

while True:
    question = input("Sua pergunta: ").strip()

    if question.lower() == "sair":
        print("Encerrando...")
        break

    if not question:
        print("Digite uma pergunta válida.\n")
        continue

    result = generator.forward(question, conn)
    print(f"SQL: {result.sql_query}")
    print(f"Resultado: {result.results if not result.error else f'Erro: {result.error}'}\n")

conn.close()
print("✓ Concluído!")

MODO INTERATIVO - digite 'sair' para encerrar

Sua pergunta: tem arroz?


2026/05/07 13:25:35 WARNING dspy.primitives.module: Calling module.forward(...) on SQLGenerator directly is discouraged. Please use module(...) instead.


In [ ]:
def sql_execution_metric(example, pred, trace=None):
    conn = sqlite3.connect("supermarket.db")

    try:
        expected_result = conn.execute(example.sql_query).fetchall()
        predicted_result = conn.execute(pred.sql_query).fetchall()

        conn.close()

        return expected_result == predicted_result

    except Exception:
        conn.close()
        return False